# Trabajo Práctico  - Calibración y Pose de la Cámara

## Resolución

In [1]:
import numpy as np 


In [2]:
from alc import * 

## GRUPO: NOMBRE DEL GRUPO

# Integrantes
* integrante
* integrante

### Ejercicio 1 - DLT ecuaciones formales

Siendo $Q\in \mathbb{R}^{2n\times 12}$, ¿qué rango debe tener para asegurar una solución no nula única a menos de un factor de escala? ¿Cuántos puntos, como mínimo, aseguran la solución y qué condición deben cumplir?

A)

Intuicion: 
+ La matriz Q tiene 2n filas porque para cada punto (pixel) tiene dos coordenadas u y v y hay n puntos
+ El 12 es porque M es de dim 3 x 4 
+ U esta en funcion de m1 y m3 entonces necesita que m2 desaparezca → los 0s deben matchear con la cantidad de elementos de m2

+ ¿Que significa que este a menos de un factor de escala?
    que no sea un solo vector, sino una dirección: <(1,0,0)>

+ Q.m = 0, Q ∈ 2n x 12, m ∈ 12 x 1, 0 ∈ 2n x 1
+ Sol es m
+ dim = cantidad de columnas de Q, cantidad de incógnitas
+ Rango es dim(Q), filas no nulas → 12 - dim(Nu(Q)) 
    + Como queremos sol no nula, el nucleo tiene que tener algo
    + Como queremos que sea única, debe tener dim(Nu) = 1 

+ Luego Rg(Q) = 12 - 1 = 11

+ rg(Q) <= 2n
+ 11 <= 2n 
+ n >= 11/2 = 5,5 ⟹ a partir de 6 puntos podemos asegurar la sol unica ≠ 0
+ Las filas de Q deben ser LI pues con 6 puntos, tenemos 12 filas y para que el rango sea 11 se nos puede ir una fila LD 

+ Condicion geometrica???


Encontrar el vector $\mathbf{m}$ que resuelve la minimización de la ecuación 6.

B) 
+ La norma 2 de una matriz A esta acotada inferiormente por su valor singular inferior y superiormente por su valor singular mayor
+ La norma 2 de A se puede definir como máx {||A v||} con ||v|| = 1, por lo que se cumple la siguiente cota:
    + σ min . ||v|| <= ||A v|| <= σ máx . ||v||
+ Como la norma de v es 1, la norma del producto queda acotada por los valores singulares (max y min)
+ Para minimizar el ||Av||, necesito conseguir que ||Av|| = σ min

+ m tiene ques ser el autovector (normalizado) asociado al menor valor singular (autovalor de Qt . Q)

### Ejercicio 2 - Calibración con puntos 3D

a) Implementar la función obtenerQ3d(imgPts, objPts) que devuelve la matriz Q a partir
de la ecuación 6

In [ ]:
def obtenerQ3d(imgPts, objPts): # asumiendo que recibimos dos arrays 
    n = objPts.shape[0] # como sabemos el tipo de datos que nos pasan????
    Q = np.zeros((2*n, 12))

    for i in range(n): 
        U= objPts[i][0]
        V= objPts[i][1]
        W= objPts[i][2]
        u = imgPts[2*i]
        v = imgPts[2*i +1]  

        Q[2*i] = np.array([U, V, W, 1, 0, 0, 0, 0, -u*U, -u*V, -u*W, -u])
        Q[2*i + 1] = np.array([0, 0, 0, 0, U, V, W, 1, -v*U, -v*V, -v*W, -v ])

    return Q 

b) A partir de lo resuelto en el Ejercicio 1 b), implementar la función resolverDLT3D(Q),
que a partir de Q encuentra la matriz de proyección M.


In [ ]:
# vamos a usar la desc de SVD pero que pasa si el valor singular menor es cero?? en ese caso no deberiamos quedarnos con ese vector? pero la reducida saca todo lo asignado a los ceros 

def resolverDLT3D(Q): 
    U, E, V = svd_reducida(Q)
    V_t = trasponer(V)
    n = V_t.shape[0]
    m = V_t[n-1]

    M = m.reshape(3, 4)

    return M 


c) Una vez recuperada M, se la descompone en la matriz de parámetros intrínsecos K,
la matriz de rotación R y el vector de traslación t, imponiendo que R sea ortonormal
(RRT = I). Para ello, implementar la función devolverMatricesCalibracion3d(M),
que devuelva R, t y K.


In [ ]:
def devolverMatricesCalibracion3d(M): 
    

a) Utilizando los datos de la carpeta $\texttt{calib/3d}$ calcular $\mathbf{R}$,  $\mathbf{t}$ y $\mathbf{K}$.

In [ ]:
#Sesgo 1 es el 0, y sesgo 2 es alfa u = alfa v
def devolverMatricesCalibracion3D(M, sesgo1=False, sesgo2=False):

    R, K = QR_con_GS(M[:,:3])

    if sesgo1 == True:
        K[0][1] = 0
    elif sesgo2 == True:
        K[1][1] = K[0][0] #nos quedamos con alfa 1, preguntar que hay que hacer con la igualdad de alfas

    columna_m = M[:,3] #chequear si slicing esta bien con el 3 (queremos la ultima columna de m)
    t = res_tri(K,columna_m)

    return R, t, K

Calcular un nuevo set de parámetros para 

b) $\alpha_u = \alpha_v$ 

c) $K_{12}=0$



In [ ]:
#Se lo agregamos a la de arriba, hay que entregar las dos?

### Ejercicio 3 - Calibración con puntos coplanares

Resolver la calibración para el caso de puntos coplanares, trabajando sobre los datos de la carpeta $\texttt{calib/coplanar}$ con las imágenes $\texttt{Im\_R\_1.png}$ a $\texttt{Im\_R\_20.png}$.


a) Obtener la matriz $\mathbf{K}$

b) Encontrar la pose de $\texttt{Im\_R\_1.png}$, $\texttt{Im\_R\_10.png}$ y $\texttt{Im\_R\_20.png}$.

### Ejercicio 4 - Fútbol

a) Calcular la homografía de la cancha de futbol de la foto de $\texttt{gol-enzo.png}$. Encontrar la distancia a la que se encontraba Enzo. 
    Dibujar sobre la fotografía círculos de 1 metro de diámetro alrededor de los jugadores argentinos.

b) Calcular la homografía de la changa de fútbol de la foto $\texttt{posibleoffside.jpg}$. 
    Trazar sobre la fotografía una recta sobre el pié del jugador del equipo defensor que defina si hay fuera de juego.
    

### Ejercicio 5 - Cruzando en rojo - Cálculo de velocidad

a) Calcular los parámetros intrínsecos de la cámara a partir de las diferentes vistas de la esquina y las coordenadas de los puntos en el archivo $\texttt{world\_img\_pts.csv}$

b) A partir de los puntos $\texttt{gs\_10\_09\_point\_correspondences\_v5.txt}$ calcular la pose de la cámara y reproyectar los puntos sobre la imagen $\texttt{img\_10\_18\_47\_351.jpg}$.

c) Calcular la velocidad de los dos primeros vehículos (taxi y camioneta) y de la señora de buzo rojo que cruza la calle. Graficar las velocidades en un plot temporal a lo largo de la secuencia.

### Ejercicio 6 - Aterrizaje suave

a) Aplicar POSIT para encontrar la pose ($\mathbf{R}$ y $\mathbf{t}$) de la cámara del Drone a partir de la imagen $\texttt{row\_00248.png}$. La cámara tiene la matriz intrínseca $\mathbf{K}$. 

b) Estimar la matriz de rotación y traslación para ubicar al drone exactamente sobre la letra H y que la cámara la vea vertical (el punto blanco marca la parte superior izquierda)

---